In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%cd /content
# 처음이면 clone, 이미 있으면 pull
!git clone https://github.com/chaeyoungwon/mask-aware-inpainting.git
# 또는
# !git checkout yeonholee
%cd mask-aware-inpainting


## 데이터셋 구성

In [ ]:
# kaggle.json 업로드
from google.colab import files
files.upload()  # kaggle.json 선택

# 설치 및 인증
!pip install kaggle
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

In [ ]:
!kaggle datasets download -d jessicali9530/celeba-dataset -p ./mask-aware-inpainting/data/
!cd mask-aware-inpainting/data && unzip celeba-dataset.zip -d celeba_temp

In [ ]:
import os

# 이미지를 올바른 위치로 이동
src = '/content/mask-aware-inpainting/mask-aware-inpainting/data/celeba/img_align_celeba/img_align_celeba'
dst = '/content/mask-aware-inpainting/data/celeba/img_align_celeba'

os.makedirs('/content/mask-aware-inpainting/data/celeba', exist_ok=True)
os.rename(src, dst)
print("완료:", len(os.listdir(dst)), "개")

base = '/content/mask-aware-inpainting/data/celeba'
filenames = sorted(os.listdir(f'{base}/img_align_celeba'))
n = len(filenames)

# list_eval_partition.txt
with open(f'{base}/list_eval_partition.txt', 'w') as f:
    for i, fname in enumerate(filenames):
        split = 0 if i < 162770 else (1 if i < 182637 else 2)
        f.write(f'{fname} {split}\n')

# identity
with open(f'{base}/identity_CelebA.txt', 'w') as f:
    for fname in filenames:
        f.write(f'{fname} 1\n')

# attr (더미)
attrs = ['5_o_Clock_Shadow','Arched_Eyebrows','Attractive','Bags_Under_Eyes','Bald','Bangs','Big_Lips','Big_Nose','Black_Hair','Blond_Hair','Blurry','Brown_Hair','Bushy_Eyebrows','Chubby','Double_Chin','Eyeglasses','Goatee','Gray_Hair','Heavy_Makeup','High_Cheekbones','Male','Mouth_Slightly_Open','Mustache','Narrow_Eyes','No_Beard','Oval_Face','Pale_Skin','Pointy_Nose','Receding_Hairline','Rosy_Cheeks','Sideburns','Smiling','Straight_Hair','Wavy_Hair','Wearing_Earrings','Wearing_Hat','Wearing_Lipstick','Wearing_Necklace','Wearing_Necktie','Young']
with open(f'{base}/list_attr_celeba.txt', 'w') as f:
    f.write(f'{n}\n{" ".join(attrs)}\n')
    for fname in filenames:
        f.write(fname + ' ' + ' '.join(['-1']*40) + '\n')

# bbox / landmarks (더미)
with open(f'{base}/list_bbox_celeba.txt', 'w') as f:
    f.write('image_id x_1 y_1 width height\n')
    for fname in filenames:
        f.write(f'{fname} 0 0 128 128\n')

with open(f'{base}/list_landmarks_align_celeba.txt', 'w') as f:
    f.write(f'{n}\nlefteye_x lefteye_y righteye_x righteye_y nose_x nose_y leftmouth_x leftmouth_y rightmouth_x rightmouth_y\n')
    for fname in filenames:
        f.write(f'{fname} 32 32 96 32 64 64 32 96 96 96\n')

print("메타파일 완료")
print(os.listdir(base))

In [ ]:
!python3 prepare_fixed_testset.py

In [ ]:
# conv_type을 'gated'으로 설정
import re
with open('config.py', 'r') as f:
    _cfg = f.read()
_cfg = re.sub(r"conv_type\s*=\s*'[^']*'", "conv_type = 'gated'", _cfg)
with open('config.py', 'w') as f:
    f.write(_cfg)
print("conv_type set to 'gated'")


## 학습

In [ ]:
!python3 train.py

In [ ]:
!python evaluate.py checkpoints/gated/best_model.pth --conv_type gated --fixed_testset ./fixed_testset


## 시각화

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv('checkpoints/gated/training_history_gated.csv')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle('Gated CAE', fontsize=14)

ax1.plot(df['epoch'], df['val_psnr'], color='steelblue')
ax1.set_title('Epoch별 Val PSNR')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('PSNR (dB)'); ax1.grid(True)

ax2.plot(df['epoch'], df['val_ssim'], color='goldenrod')
ax2.set_title('Epoch별 Val SSIM')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('SSIM'); ax2.grid(True)

plt.tight_layout()
plt.show()
print(f'최종 PSNR: {df["val_psnr"].iloc[-1]:.2f} dB  |  최종 SSIM: {df["val_ssim"].iloc[-1]:.4f}')
